# Week 12b: Deep Research Agents
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

This notebook is the **key differentiator** lab for *Applied Generative AI*: you will **build** a small autonomous research agent—conceptually aligned with products like **ChatGPT Deep Research**, **Gemini Deep Research**, and **Perplexity**—using **Semantic Scholar** (free API), **LangGraph**, and either **open-source Flan-T5** or an optional **OpenAI** model for language-heavy steps.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand how deep research agents work** — Search, extract, synthesize, and report in an iterative loop.
2. **Build an autonomous research agent** — Implement iterative search with LangGraph and real academic search.
3. **Implement source attribution and citation generation** — Track provenance, emit inline citations, and build a references section.
4. **Design stopping criteria for research depth** — Max iterations, quality thresholds, and diminishing returns.
5. **Evaluate research agent output quality** — Coverage, accuracy (citation grounding), coherence, and a simple scorecard.

---
## Setup — Install & imports

We use **LangGraph** for orchestration, **requests** for Semantic Scholar, **transformers** for Flan-T5 (no API key), and optional **OpenAI** for stronger extraction/synthesis. **sentence-transformers** and **faiss-cpu** are included for optional local retrieval extensions you might explore in exercises.

In [ ]:
%pip install -q langchain langchain-core langchain-openai langgraph openai transformers sentence-transformers faiss-cpu requests beautifulsoup4


In [ ]:
import os
import re
import requests
from getpass import getpass
from typing import TypedDict, Literal, Annotated
from operator import add

if not os.environ.get("OPENAI_API_KEY"):
    try:
        key = getpass("OpenAI API key (optional — press Enter to skip): ")
        if key.strip():
            os.environ["OPENAI_API_KEY"] = key.strip()
    except Exception:
        pass

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OpenAI configured: {HAS_OPENAI}")
if not HAS_OPENAI:
    print("Using Flan-T5-base for extraction/synthesis (local, open weights).")

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

S2_HEADERS = {"User-Agent": "AppliedGenAI-Course/1.0 (educational; contact: instructor)"}

if HAS_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    pipe = None
else:
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_length=512)

    class FlanT5Wrapper:
        def invoke(self, messages):
            text = messages[-1].content if hasattr(messages[-1], "content") else str(messages[-1])
            out = pipe(text, max_new_tokens=256)[0]["generated_text"]
            return type("Msg", (), {"content": out})()

    llm = FlanT5Wrapper()

---
## 1. What is a Deep Research Agent?

### How commercial systems work (conceptually)

| System | User experience | Under the hood (typical) |
|--------|-----------------|---------------------------|
| **ChatGPT Deep Research** | Long-running report with citations | Multi-step browsing / retrieval, chunking, synthesis, revision |
| **Gemini Deep Research** | Structured report over many sources | Search + tool use + planning + verification passes |
| **Perplexity** | Answer + inline sources | Retrieval, ranking, concise synthesis, citation cards |

None of these magically “knows the web”—they **retrieve**, **read**, **compress**, and **compose**, usually in a **loop** with **stopping rules**.

### Architecture: Query → Plan → Search → Extract → Synthesize → Iterate → Report

```
                    ┌──────────────┐
                    │    Query     │
                    └──────┬───────┘
                           ▼
                    ┌──────────────┐
                    │    Plan      │  (sub-questions, facets, keywords)
                    └──────┬───────┘
                           ▼
   ┌──────────┐     ┌──────────────┐     ┌─────────────┐
   │  Search  │────▶│   Extract    │────▶│  Synthesize │
   └──────────┘     └──────────────┘     └──────┬──────┘
        ▲                                        │
        │         ┌──────────────────────────────┘
        │         ▼
        │   ┌──────────────┐      No
        └───│ Good enough? │────────────▶ Report + citations
            └──────────────┘
                  │ Yes (need more)
                  ▼
            Refined search / gap fill
```

### Key decisions

- **Search deeper** when coverage is thin, claims conflict, or important facets (methods, limitations, dates) are missing.
- **Stop** when a **quality bar** is met, **max iterations** is reached, or **diminishing returns** set in (new sources repeat known facts).

---
## 2. Building the research pipeline

We define a **typed state**, implement **search** (Semantic Scholar), **extraction** (structured bullets with source tags), and **synthesis** (narrative answer), then compile a **linear** LangGraph for a single pass before we add iteration.

In [ ]:
class ResearchState(TypedDict):
    query: str
    search_results: Annotated[list, add]
    extracted_info: list
    synthesis: str
    iteration: int
    should_continue: bool
    quality_score: float
    prev_quality: float
    report: str
    references: list
    last_search_query: str

In [ ]:
def search_semantic_scholar(query: str, limit: int = 5) -> list[dict]:
    """Semantic Scholar Graph API — free tier, no API key (please use politely)."""
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": limit,
        "fields": "paperId,title,authors,year,abstract,url",
    }
    try:
        r = requests.get(url, params=params, headers=S2_HEADERS, timeout=15)
        r.raise_for_status()
        papers = r.json().get("data", []) or []
        out = []
        for p in papers:
            authors = p.get("authors") or []
            names = ", ".join(a.get("name", "") for a in authors[:4])
            out.append(
                {
                    "paperId": p.get("paperId") or "",
                    "title": p.get("title") or "",
                    "authors": names,
                    "year": p.get("year", ""),
                    "abstract": (p.get("abstract") or "") or "(No abstract available)",
                    "url": p.get("url") or "",
                }
            )
        return out
    except Exception as e:
        return [
            {
                "paperId": "",
                "title": "(Search error)",
                "authors": "",
                "year": "",
                "abstract": str(e)[:400],
                "url": "",
            }
        ]

In [ ]:
demo_hits = search_semantic_scholar("transformer self-attention neural machine translation", limit=3)
for i, p in enumerate(demo_hits, 1):
    print(f"{i}. {p['title'][:70]}…  ({p.get('year','')})")
    print(f"   id={p.get('paperId','')[:16]}…  {p['abstract'][:110]}…\n")

In [ ]:
def extract_from_results(state: ResearchState) -> dict:
    """Extract key claims; keep provenance as [Source k] aligned with search order."""
    results = state.get("search_results") or []
    if not results:
        return {"extracted_info": []}

    blocks = []
    for i, p in enumerate(results):
        tag = f"[Source {i + 1}] {p.get('title', '')} ({p.get('year', '')})"
        blocks.append(f"{tag}\n{p.get('abstract', '')}")
    context = "\n\n".join(blocks)

    prompt = f"""Research question: {state['query']}

From the sources below, list 3-5 concise facts useful for answering the question.
Each fact must start with the tag [Source N] matching the source header.
One fact per line.

{context}

Facts:"""

    if HAS_OPENAI:
        text = llm.invoke([HumanMessage(content=prompt)]).content
    else:
        text = pipe(prompt, max_new_tokens=320)[0]["generated_text"]

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    extracted = []
    for ln in lines:
        if "[Source" in ln:
            extracted.append({"fact": ln, "raw": ln})
    if not extracted:
        extracted = [{"fact": text[:400], "raw": text}]

    return {"extracted_info": extracted[:8]}

In [ ]:
def synthesize(state: ResearchState) -> dict:
    """Merge extracted facts into a short report draft (citations preserved as [Source i])."""
    facts = "\n".join(e.get("fact", e.get("raw", "")) for e in (state.get("extracted_info") or []))
    query = state["query"]
    prev = state.get("synthesis") or ""

    prompt = f"""Question: {query}

Bullet facts from literature:
{facts}

Write 2-4 short paragraphs answering the question. Keep inline tags like [Source 1] where a claim is supported.
"""
    if prev:
        prompt += f"\nPrior draft (revise/improve, do not drop cited claims):\n{prev}\n"

    if HAS_OPENAI:
        out = llm.invoke([HumanMessage(content=prompt)]).content
    else:
        out = pipe(prompt, max_new_tokens=400)[0]["generated_text"]

    return {"synthesis": out}

In [ ]:
def search_node(state: ResearchState) -> dict:
    q = state.get("last_search_query") or state["query"]
    hits = search_semantic_scholar(q, limit=5)
    return {"search_results": hits, "last_search_query": q}


linear = StateGraph(ResearchState)
linear.add_node("search", search_node)
linear.add_node("extract", extract_from_results)
linear.add_node("synthesize", synthesize)
linear.set_entry_point("search")
linear.add_edge("search", "extract")
linear.add_edge("extract", "synthesize")
linear.add_edge("synthesize", END)

research_pipeline = linear.compile()

seed = {
    "query": "What are the main innovations in the Transformer architecture?",
    "search_results": [],
    "extracted_info": [],
    "synthesis": "",
    "iteration": 0,
    "should_continue": False,
    "quality_score": 0.0,
    "prev_quality": 0.0,
    "report": "",
    "references": [],
    "last_search_query": "",
}

one_pass = research_pipeline.invoke(seed)
print(one_pass["synthesis"][:900])

---
## 3. Iterative refinement

A **deep research** agent must decide: *Is my evidence sufficient?* We implement a lightweight **quality check** (coverage, depth, consistency proxy), combine it with **max iterations**, and add **diminishing returns**: if the score barely improves versus the last round, stop even if below the ideal threshold.

In [ ]:
def consistency_proxy(text: str) -> float:
    """Very rough signal: repeated contradictory markers (demo heuristic, not NLP gold)."""
    t = text.lower()
    neg = sum(t.count(w) for w in ["however", "contrary", "unclear", "debate", "disagree"])
    return max(0.0, 1.0 - min(0.3, neg * 0.06))


def quality_check(state: ResearchState, max_iterations: int = 3) -> dict:
    text = state.get("synthesis") or ""
    it = int(state.get("iteration") or 0)
    words = len(text.split())
    tags = len(re.findall(r"\[Source\s*\d+\]", text))
    coverage = min(1.0, words / 160.0)
    depth = min(1.0, tags / 3.0)
    consist = consistency_proxy(text)
    score = 0.45 * coverage + 0.35 * depth + 0.20 * consist

    prev = float(state.get("prev_quality") or 0.0)
    delta = score - prev
    quality_threshold = 0.62
    min_delta = 0.03

    stop_good = score >= quality_threshold
    stop_cap = it >= max_iterations - 1
    stop_diminishing = (it >= 1) and (delta < min_delta)

    should_continue = not (stop_good or stop_cap or stop_diminishing)

    return {
        "quality_score": float(score),
        "prev_quality": float(score),
        "should_continue": bool(should_continue),
    }


def quality_check_node(state: ResearchState) -> dict:
    return quality_check(state)

In [ ]:
def follow_up_query(state: ResearchState) -> str:
    base = state["query"]
    syn = (state.get("synthesis") or "")[:500]
    prompt = f"""Main question: {base}

Draft answer so far:
{syn}

Reply with ONE short academic search query (5-12 words) to find missing evidence."""
    if HAS_OPENAI:
        return llm.invoke([HumanMessage(content=prompt)]).content.strip().splitlines()[0][:120]
    return f"{base} limitations evaluation survey"


def decide_next_search(state: ResearchState) -> dict:
    if not state.get("should_continue"):
        return {}
    fq = follow_up_query(state)
    hits = search_semantic_scholar(fq, limit=4)
    return {
        "search_results": hits,
        "last_search_query": fq,
        "iteration": int(state.get("iteration") or 0) + 1,
    }


def route_after_quality(state: ResearchState) -> Literal["more_research", "finalize"]:
    return "more_research" if state.get("should_continue") else "finalize"


def route_after_more(state: ResearchState) -> Literal["extract", "finalize"]:
    return "extract" if state.get("should_continue") else "finalize"

In [ ]:
def finalize_placeholder(state: ResearchState) -> dict:
    return {"report": state.get("synthesis", "")}


iter_graph = StateGraph(ResearchState)
iter_graph.add_node("search", search_node)
iter_graph.add_node("extract", extract_from_results)
iter_graph.add_node("synthesize", synthesize)
iter_graph.add_node("quality_check", quality_check_node)
iter_graph.add_node("more_research", decide_next_search)
iter_graph.add_node("finalize", finalize_placeholder)

iter_graph.set_entry_point("search")
iter_graph.add_edge("search", "extract")
iter_graph.add_edge("extract", "synthesize")
iter_graph.add_edge("synthesize", "quality_check")
iter_graph.add_conditional_edges(
    "quality_check",
    route_after_quality,
    {"more_research": "more_research", "finalize": "finalize"},
)
iter_graph.add_conditional_edges(
    "more_research",
    route_after_more,
    {"extract": "extract", "finalize": "finalize"},
)
iter_graph.add_edge("finalize", END)

iterative_agent = iter_graph.compile()

run = iterative_agent.invoke(seed)
print("Iterations (0-based counter):", run.get("iteration"), "| quality:", round(run.get("quality_score", 0), 3))
print(run.get("report", run.get("synthesis"))[:700])

---
## 4. Source attribution, inline citations, and verification

**Attribution** means every non-trivial claim can be traced to a retrieved document. We map `[Source i]` → numeric `[i]` for readability, append a **References** section, and **verify** that `paperId` values still resolve in Semantic Scholar (guards against stale or hallucinated IDs in real systems you would cross-check similarly).

In [ ]:
def build_references(state: ResearchState) -> list[dict]:
    refs, seen = [], set()
    for p in state.get("search_results") or []:
        key = p.get("paperId") or p.get("title") or ""
        if not key or key in seen:
            continue
        seen.add(key)
        rid = len(refs) + 1
        refs.append(
            {
                "id": rid,
                "paperId": p.get("paperId", ""),
                "title": p.get("title", ""),
                "authors": p.get("authors", ""),
                "year": p.get("year", ""),
                "url": p.get("url", ""),
                "citation": f"{p.get('authors','')}. ({p.get('year','')}). {p.get('title','')}. {p.get('url','')}",
            }
        )
    return refs

In [ ]:
def generate_report_with_citations(state: ResearchState) -> dict:
    body = state.get("synthesis") or ""
    refs = build_references(state)
    report = body
    for i in range(1, len(refs) + 1):
        report = re.sub(rf"\[Source\s*{i}\]", f"[{i}]", report, flags=re.IGNORECASE)
    bib = "\n\n## References\n\n" + "\n".join(f"[{r['id']}] {r['citation']}" for r in refs)
    return {"report": report + bib, "references": refs}


def verify_paper(paper_id: str) -> bool:
    if not paper_id:
        return False
    url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}"
    try:
        r = requests.get(url, params={"fields": "title"}, headers=S2_HEADERS, timeout=8)
        return r.status_code == 200
    except Exception:
        return False


chk = search_semantic_scholar("BERT pretraining deep bidirectional transformers", limit=2)
st = {**seed, "search_results": chk, "synthesis": "Bidirectional context matters [Source 1]."}
rep = generate_report_with_citations(st)
print(rep["report"][:400], "…")
for r in rep["references"]:
    print("verify", r["id"], verify_paper(r["paperId"]))

---
## 5. Evaluating research agent output

We use an **automatic scorecard** (transparent heuristics) to discuss **coverage** (breadth/length), **accuracy** (citation density + reference count), and **coherence** (structure). This does **not** replace human peer review—but it gives students a quantitative lens for iteration.

In [ ]:
def evaluate_research_report(report: str, query: str, references: list) -> dict:
    scores = {}
    words = len(report.split())
    paras = [p for p in report.split("\n\n") if p.strip()]
    scores["coverage"] = round(min(1.0, 0.5 * min(1.0, words / 220) + 0.5 * min(1.0, len(paras) / 4)), 2)

    cite_markers = len(re.findall(r"\[\d+\]", report))
    scores["accuracy"] = round(
        min(
            1.0,
            0.55 * min(1.0, cite_markers / 4) + 0.45 * min(1.0, len(references) / 4),
        ),
        2,
    )

    sents = [s for s in re.split(r"(?<=[.!?])\s+", report) if len(s) > 20]
    scores["coherence"] = round(min(1.0, len(sents) / 6), 2)
    scores["overall"] = round(sum(scores[k] for k in ("coverage", "accuracy", "coherence")) / 3, 2)
    return scores

In [ ]:
def print_scorecard(scores: dict) -> None:
    print("Research Quality Scorecard")
    print("-" * 34)
    for k, v in scores.items():
        bar = "█" * int(v * 10) + "░" * (10 - int(v * 10))
        print(f"  {k:12} {bar} {v:.2f}")


demo = "Transformers rely on self-attention [1]. BERT uses bidirectional context [2].\n\nMore work explores scaling [1]."
print_scorecard(evaluate_research_report(demo, "Transformers", [{"id": 1}, {"id": 2}]))

---
## 6. The full deep research agent

We swap the placeholder finalizer for **citation rendering**, then **stream** the graph to visualize the trajectory: **search → extract → synthesize → quality → (optional) more research → finalize**.

In [ ]:
def finalize_report_node(state: ResearchState) -> dict:
    return generate_report_with_citations(state)


full_graph = StateGraph(ResearchState)
full_graph.add_node("search", search_node)
full_graph.add_node("extract", extract_from_results)
full_graph.add_node("synthesize", synthesize)
full_graph.add_node("quality_check", quality_check_node)
full_graph.add_node("more_research", decide_next_search)
full_graph.add_node("finalize", finalize_report_node)

full_graph.set_entry_point("search")
full_graph.add_edge("search", "extract")
full_graph.add_edge("extract", "synthesize")
full_graph.add_edge("synthesize", "quality_check")
full_graph.add_conditional_edges(
    "quality_check",
    route_after_quality,
    {"more_research": "more_research", "finalize": "finalize"},
)
full_graph.add_conditional_edges(
    "more_research",
    route_after_more,
    {"extract": "extract", "finalize": "finalize"},
)
full_graph.add_edge("finalize", END)

deep_research_agent = full_graph.compile()

In [ ]:
sample_query = "What are the key differences between BERT and GPT-style architectures?"
start = {
    "query": sample_query,
    "search_results": [],
    "extracted_info": [],
    "synthesis": "",
    "iteration": 0,
    "should_continue": False,
    "quality_score": 0.0,
    "prev_quality": 0.0,
    "report": "",
    "references": [],
    "last_search_query": "",
}

print("=== Trajectory (stream) ===")
for step in deep_research_agent.stream(start):
    for node, update in step.items():
        print(f"\n→ {node}")
        if not isinstance(update, dict):
            continue
        for k, v in update.items():
            if k == "search_results" and isinstance(v, list):
                print(f"   {k}: {len(v)} papers")
            elif k == "extracted_info" and isinstance(v, list):
                print(f"   {k}: {len(v)} bullets")
            elif k == "synthesis" and isinstance(v, str):
                print(f"   {k}: {len(v)} chars")
            elif k in ("quality_score", "should_continue", "iteration", "last_search_query"):
                print(f"   {k}: {v}")

final = deep_research_agent.invoke(start)
print("\n=== Final report (excerpt) ===")
print(final.get("report", "")[:1200])

In [ ]:
print_scorecard(
    evaluate_research_report(
        final.get("report", ""),
        sample_query,
        final.get("references", []),
    )
)

---
## 7. Exercises

**Exercise 1 — Quality-first stopping.**  
Edit `quality_check` to prioritize a **quality threshold** (e.g. 0.72) and use **max iterations** only as a safety cap. Log each round’s `quality_score` and whether stopping was due to threshold, cap, or diminishing returns.

**Exercise 2 — Gap identification.**  
Insert a node **before** `more_research` that writes an explicit *gap list* (e.g. “missing: quantitative benchmarks”, “missing: failure modes”). Use that list to steer `follow_up_query` (template or LLM).

**Exercise 3 — Comparative evaluation.**  
 Run `deep_research_agent.invoke` on **two** topics (e.g. *RAG vs fine-tuning for domain adaptation* and *fairness risks of automated hiring*). Compare scorecards; manually read whether high scores match human judgment.

**Stretch — Local retrieval.**  
 Embed abstracts with `sentence-transformers`, store in FAISS, and retrieve top‑k chunks before extraction.

---
## 8. Responsible AI reflection

Deep research agents can synthesize dozens of sources in minutes, but they can also **fabricate citations**, **cherry-pick supporting evidence**, and present synthesis with **false confidence**. As these tools become more powerful, how do we maintain the intellectual rigor that human researchers develop through years of training?

> **Is there a risk that AI research agents will produce "good enough" research that displaces deeper human inquiry?**

Discuss: **verification** (spot-check primary sources), **adversarial retrieval** (seek disconfirming evidence), **transparent limitations** sections, and **human review** for high-stakes decisions.

---
## 9. Summary & next steps

**Takeaways**
- **Pipeline:** Plan/search → extract → synthesize → **quality gate** → iterate or finalize.
- **Semantic Scholar** provides a realistic, keyless academic search surface; always attribute with `paperId` / URLs.
- **Stopping** blends **thresholds**, **budgets**, and **diminishing returns**.
- **Evaluation** scorecards are teaching tools—pair them with critical reading.

**Next (Week 12c)**  
Reasoning-and-planning agents: **ReAct**, explicit **tool use**, and richer **planner–executor** graphs building on this research loop.